In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import xml.sax
import random
import os


input_filename = "/content/drive/MyDrive/input.xml"
output_filename = "/content/drive/MyDrive/output_anonymized.xml"


names_pool = ["Иван", "Петр", "Сидор", "Анна", "Мария", "Алексей", "Елена"]
surnames_pool = ["Иванов", "Петров", "Сидоров", "Смирнова", "Кузнецова", "Попов"]
fruits_pool = ["Яблоко", "Груша", "Слива", "Вишня", "Абрикос", "Банан", "Апельсин", "Огурец", "Помидор"]

mapping_dict = {}

def get_fake_name(original_name):
    if original_name in mapping_dict:
        return mapping_dict[original_name]

    orig_lower = original_name.lower()

    object_keywords = ["grid", "network", "center", "substation", "office", "rdo", "res", "пс", "тп", "лэп"]

    if any(keyword in orig_lower for keyword in object_keywords):
        fake = f"{random.choice(fruits_pool)} electrical network"
    else:
        fake = f"{random.choice(surnames_pool)} {random.choice(names_pool)}"

    mapping_dict[original_name] = fake
    return fake

class CIMAnonymizer(xml.sax.ContentHandler):
    def __init__(self, output_file):
        super().__init__()
        self.output_file = output_file
        self.in_name_tag = False

    def startDocument(self):
        self.output_file.write('<?xml version="1.0" encoding="utf-8"?>\n')

    def startElement(self, name, attrs):
        self.output_file.write(f'<{name}')
        for attr_name, attr_value in attrs.items():
            self.output_file.write(f' {attr_name}="{attr_value}"')
        self.output_file.write('>')

        if name == 'cim:IdentifiedObject.name':
            self.in_name_tag = True

    def characters(self, content):
        if self.in_name_tag and content.strip():
            fake_content = get_fake_name(content.strip())
            self.output_file.write(fake_content)
        else:
            self.output_file.write(content)

    def endElement(self, name):
        self.output_file.write(f'</{name}>\n')
        if name == 'cim:IdentifiedObject.name':
            self.in_name_tag = False


if not os.path.exists(input_filename):
    print(f"ОШИБКА: Файл не найден по пути {input_filename}")
    print("Проверьте, что вы загрузили его на Google Диск в корень (не в папку) и имя точно 'input.xml'")
else:
    print("Начинаю обработку файла. Это займет пару минут, не закрывайте вкладку...")

    with open(input_filename, 'r', encoding='utf-8') as f_in, \
         open(output_filename, 'w', encoding='utf-8') as f_out:

        parser = xml.sax.make_parser()
        parser.setContentHandler(CIMAnonymizer(f_out))
        parser.parse(f_in)

    print(f"УСПЕХ! Файл сохранен на вашем Google Диске как {output_filename}")
    print(f"Заменено уникальных имен: {len(mapping_dict)}")


Начинаю обработку файла. Это займет пару минут, не закрывайте вкладку...
УСПЕХ! Файл сохранен на вашем Google Диске как /content/drive/MyDrive/output_anonymized.xml
Заменено уникальных имен: 120210
